# Section 4: The Heart of the Matter — Attention

This is the central section of the tutorial. Everything else is plumbing around what we build here.

**The problem:** after embedding, each token has a vector, but those vectors are context-free. The vector for "it" in "The animal didn't cross the street because **it** was too tired" is the same whether "it" refers to "animal" or "street". The model needs a mechanism to let tokens look at each other and update their representations based on context.

**The solution:** attention. Each token produces a weighted average of all other tokens' vectors, where the weights are computed from similarity — tokens that are relevant to each other get high weight, irrelevant ones get low weight.

We build this in 7 steps, each a small addition to the previous one:
1. **4.1** — the intuition: weighted averages
2. **4.2** — dot product as a similarity measure
3. **4.3** — softmax to turn scores into weights
4. **4.4** — naive self-attention (no learned parameters yet)
5. **4.5** — queries, keys, values (learned projections)
6. **4.6** — scaling to keep softmax stable
7. **4.7** — visualising the attention matrix as a heatmap

## Setup

We reuse the tokenizer and embedding code from the previous sections to get a sequence embedding matrix `X` of shape `(seq_len, d_model)` — the starting point for everything in this section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tokenizer import CharTokenizer

D_MODEL = 8  # dimensionality of every token vector, kept small so we can print it all

# Fixed seed: makes every random number identical across runs, essential for debugging.
rng = np.random.default_rng(seed=42)

sentence = "the cat sat on the mat"
tok = CharTokenizer(sentence)
ids = tok.encode(sentence)

# Build the embedding matrix and look up embeddings for the sentence.
# Shape: E is (vocab_size, d_model), X is (seq_len, d_model).
scale = 1.0 / np.sqrt(D_MODEL)
E = rng.normal(0.0, scale, size=(tok.vocab_size, D_MODEL))
X = E[ids]  # fancy indexing: select one row per token ID

print(f"Sentence : {sentence!r}")
print(f"Token IDs: {ids}")
print(f"X shape  : {X.shape}  → (seq_len={len(ids)}, d_model={D_MODEL})")
print()
print("X (first 4 rows — first 4 characters):")
for i in range(4):
    print(f"  pos {i} ({sentence[i]!r}): {X[i].round(3)}")

## 4.1 — The Intuition: Weighted Averages

Recall the problem: the vector for "it" in "The animal didn't cross the street because **it** was too tired" doesn't know it refers to "animal". We want to update "it"'s vector by mixing in information from the other tokens — especially "animal".

The simplest way to mix: **weighted average**. Replace each token's vector with a weighted sum of all token vectors, where the weights say how much each token should contribute.

```
output[i] = sum over j of (weight[i,j] * X[j])
```

- `weight[i, j]` is how much token `i` attends to token `j`.
- Each row of weights must sum to 1 (it's a proper average, not a free sum).
- The weights must be *computed from the data*, not hard-coded.

The only question remaining: **how do we compute the weights?** That's what steps 4.2 and 4.3 answer.

## 4.2 — Dot Product as Similarity

We need a number that says "how relevant is token j to token i?". The standard answer for vectors is the **dot product**:

```
similarity(u, v) = u · v = u[0]*v[0] + u[1]*v[1] + ... + u[d-1]*v[d-1]
```

Geometrically: `u · v = |u| |v| cos(θ)`. Two vectors pointing in the same direction → large positive value. Orthogonal → zero. Opposite → large negative.

If we stack all token vectors into `X` (shape `T × d`), then `X @ X.T` gives us every pairwise dot product in one matrix multiply — entry `(i, j)` is the similarity of token `i` with token `j`. This `(T, T)` matrix is the **raw similarity matrix**.

In [ ]:
# X @ X.T: (T, d) @ (d, T) = (T, T)
# Entry S[i, j] = dot product of X[i] and X[j] = raw similarity score.
S = X @ X.T

print(f"X shape : {X.shape}  (seq_len, d_model)")
print(f"S shape : {S.shape}  (seq_len, seq_len) — one score per token pair")
print()

# Print the similarity row for position 0 ('t').
# Characters that share the same embedding (other 't's) will have the
# highest similarity because they are literally the same vector.
print(f"Raw similarity scores from position 0 ('{sentence[0]}') to every other token:")
for j, ch in enumerate(sentence):
    bar = "█" * int(max(0, S[0, j]) * 30)  # visual bar proportional to score
    print(f"  pos {j:2d} ({ch!r}): {S[0, j]:+.3f}  {bar}")

print()
print(f"Problem: scores are unbounded real numbers — some negative, no constraint.")
print(f"We need weights that are non-negative and sum to 1. Enter: softmax.")

## 4.3 — Softmax: Turning Scores into Weights

Softmax converts a vector of arbitrary real numbers into a **probability distribution** (non-negative, sums to 1):

$$\text{softmax}(s)_i = \frac{e^{s_i}}{\sum_j e^{s_j}}$$

Three key properties:
1. **Non-negative** — `exp` is always positive.
2. **Sums to 1** — the denominator enforces this.
3. **Sharpening** — larger scores get disproportionately larger weights due to the exponential. A score of 5 vs 4 produces weights of ~0.73 vs ~0.27.

**Numerical trick:** subtract the row maximum before exponentiating. This is mathematically identical (the constant cancels) but prevents `exp` from overflowing on large inputs.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    # Subtract the max along the target axis before exponentiating.
    # This is pure numerical stability — the result is mathematically identical
    # to the naive formula because the constant cancels in numerator and denominator.
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    # Divide each element by the sum of its row, so the row sums to exactly 1.
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


# --- Demonstrate the sharpening effect ---
raw = np.array([1.0, 2.0, 3.0, 4.0])

print("Effect of score magnitude on the output distribution:")
print()
for scale, label in [(0.1, "×0.1 (tiny differences → near-uniform)"),
                     (1.0, "×1.0 (baseline)"),
                     (5.0, "×5.0 (large differences → very peaked)")]:
    weights = softmax(raw * scale)
    bar = "  " + " | ".join(f"{w:.2f}" for w in weights)
    print(f"  scores {raw * scale}  →  softmax: {bar}   (sum={weights.sum():.4f})")

print()
print("Key insight: the same *pattern* of scores (1,2,3,4) produces very different")
print("distributions depending on their magnitude. This is why scaling matters (step 4.6).")

## 4.4 — Naive Self-Attention

We now have all the pieces. Attention in three lines:

1. **Scores** — `S = X @ X.T` — raw dot-product similarity between every pair.
2. **Weights** — `A = softmax(S)` — turn scores into a probability distribution per row.
3. **Output** — `Y = A @ X` — each output row is a weighted average of all input rows.

`Y` has the same shape as `X`: `(T, d_model)`. Every token now holds a mixture of all tokens' information, weighted by relevance. This is self-attention.

In [ ]:
def naive_self_attention(X: np.ndarray):
    # Step 1: dot product of every token with every other token.
    # Shape: (T, d) @ (d, T) = (T, T). Entry [i,j] = how similar token i is to token j.
    scores = X @ X.T

    # Step 2: softmax row-wise so each token's weights sum to 1.
    # axis=-1 means we normalise across the columns (the "which token to attend to" axis).
    A = softmax(scores, axis=-1)

    # Step 3: weighted average. (T, T) @ (T, d) = (T, d).
    # Row i of Y is a blend of all rows of X, weighted by how much token i attends to each.
    Y = A @ X

    return Y, A


Y, A = naive_self_attention(X)

print(f"Input  X shape : {X.shape}  (seq_len, d_model)")
print(f"Output Y shape : {Y.shape}  (seq_len, d_model) — same shape, context-enriched")
print(f"Weights A shape: {A.shape}  (seq_len, seq_len)")
print()

# Sanity check: every row of A must sum to 1.0 (it's a probability distribution).
row_sums = A.sum(axis=-1)
print(f"Row sums of A (must all be 1.0): {row_sums.round(4)}")
print()

# Show where token 0 ('t') puts its attention.
print(f"Attention from pos 0 ('{sentence[0]}') to every other position:")
for j, ch in enumerate(sentence):
    bar = "█" * int(A[0, j] * 80)
    print(f"  pos {j:2d} ({ch!r}): {A[0, j]:.3f}  {bar}")

## 4.5 — Queries, Keys, and Values

Naive attention has a structural problem: the same vector `X[i]` plays **three distinct roles** at once:
- **Query** — "what am I looking for?"
- **Key** — "what do I offer for matching against?"
- **Value** — "what content do I contribute if I'm selected?"

Tying all three roles to the same vector is too rigid. The fix: three separate learned linear projections `W_Q`, `W_K`, `W_V`, each of shape `(d_model, d_k)`:

```
Q = X @ W_Q   # queries: what each token is searching for
K = X @ W_K   # keys:    what each token advertises for matching
V = X @ W_V   # values:  what each token contributes when selected
```

The attention formula becomes: `A = softmax(Q @ K.T)`, then `Y = A @ V`.

**Analogy:** think of `Q @ K.T` as a soft database lookup. The query is your search string, each key is a document title, and the dot product is the relevance score. Softmax selects softly. The value is the document content returned.

In [ ]:
D_K = D_MODEL  # query/key dimension; with one head, simplest choice is d_model
D_V = D_MODEL  # value dimension

# Initialise the three projection matrices with the same scale trick as embeddings.
proj_scale = 1.0 / np.sqrt(D_MODEL)
W_Q = rng.normal(0.0, proj_scale, size=(D_MODEL, D_K))
W_K = rng.normal(0.0, proj_scale, size=(D_MODEL, D_K))
W_V = rng.normal(0.0, proj_scale, size=(D_MODEL, D_V))

print(f"W_Q shape: {W_Q.shape}  (d_model → d_k)")
print(f"W_K shape: {W_K.shape}  (d_model → d_k)")
print(f"W_V shape: {W_V.shape}  (d_model → d_v)")
print()

# Project the same input X three different ways — three different "views" of each token.
Q = X @ W_Q  # (T, d_k): what each token is querying for
K = X @ W_K  # (T, d_k): what each token offers as a key
V = X @ W_V  # (T, d_v): what each token contributes as a value

print(f"Q shape: {Q.shape}  — one query vector per token")
print(f"K shape: {K.shape}  — one key vector per token")
print(f"V shape: {V.shape}  — one value vector per token")
print()

# Compute attention using Q and K for scoring, V for output content.
scores_qk = Q @ K.T           # (T, T): query-key dot products
A_qk      = softmax(scores_qk, axis=-1)  # (T, T): attention weights
Y_qk      = A_qk @ V          # (T, d_v): context-mixed output

print(f"Output Y shape: {Y_qk.shape}  (same as input — shape is preserved)")
print(f"Row sums of A  (must be 1.0): {A_qk.sum(axis=-1).round(4)}")
print()
print("With W_Q, W_K, W_V the model can learn three independent roles per token.")
print("Before: attention pattern forced by raw embedding similarity.")
print("Now   : attention pattern shaped by three learnable projections.")

## 4.6 — Scaled Dot-Product Attention

There's a hidden problem: as `d_k` grows, the dot products `Q[i] · K[j]` grow in magnitude. Each dot product sums `d_k` terms, so its variance scales with `d_k`. For `d_k = 64`, scores are typically ~8× larger than for `d_k = 1`.

**Why is large magnitude bad?** Recall the sharpening effect from 4.3. When scores are very large, softmax saturates — one position gets nearly all the weight and gradients effectively vanish. The model can't learn.

**The fix:** divide scores by `√d_k` before softmaxing. This keeps the typical score magnitude around O(1) regardless of dimension.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

This is the canonical formula from *Attention Is All You Need*. It's the final form of the attention primitive — we will use it unchanged for the rest of the tutorial.

In [ ]:
def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: np.ndarray | None = None,
):
    d_k = Q.shape[-1]  # number of query/key dimensions

    # Raw scores: (T_q, d_k) @ (d_k, T_k) = (T_q, T_k).
    scores = Q @ K.T

    # THE key line: divide by sqrt(d_k) to keep score magnitudes ~O(1).
    # Without this, larger d_k means larger scores, which saturates softmax.
    scores = scores / np.sqrt(d_k)

    # Optional mask: set masked positions to a large negative number so
    # softmax gives them weight ~0. Used later for causal (decoder) masking.
    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    A = softmax(scores, axis=-1)  # (T_q, T_k): row-normalised attention weights
    Y = A @ V                     # (T_q, d_v): weighted sum of value vectors
    return Y, A


# --- Show why the scaling matters ---
print("Comparing unscaled vs scaled scores for our sentence:")
scores_unscaled = Q @ K.T
scores_scaled   = Q @ K.T / np.sqrt(D_K)

print(f"  Unscaled — mean abs score: {np.abs(scores_unscaled).mean():.4f},  std: {scores_unscaled.std():.4f}")
print(f"  Scaled   — mean abs score: {np.abs(scores_scaled).mean():.4f},  std: {scores_scaled.std():.4f}")
print()

# Run the full scaled attention.
Y_scaled, A_scaled = scaled_dot_product_attention(Q, K, V)

print(f"Output shape : {Y_scaled.shape}")
print(f"Row sums of A: {A_scaled.sum(axis=-1).round(4)}  (must all be 1.0)")

## 4.7 — Visualising the Attention Matrix

The attention matrix `A` is a `(T, T)` grid of weights. The natural way to inspect it is a **heatmap**: rows are queries ("which token is looking"), columns are keys ("which token is being looked at"). Brighter cells = more attention.

With random weights the pattern is noise — that's expected. The value of looking at it *now* is to confirm the mechanism is working and to build the habit. After training (Section 13), these heatmaps become interpretable and reveal what the model learned.

In [ ]:
# Token labels: use '_' for space so it's visible on the plot axes.
labels = [('_' if ch == ' ' else ch) for ch in sentence]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (A_plot, title) in zip(axes, [
    (A,        "Naive attention\n(no Q/K/V projections)"),
    (A_scaled, "Scaled dot-product attention\n(with learned Q/K/V projections)"),
]):
    im = ax.imshow(A_plot, cmap="viridis", vmin=0, vmax=A_plot.max())

    # Label both axes with the characters so we can read off "token i attends to token j".
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Key (token being attended to)")
    ax.set_ylabel("Query (token doing the attending)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Attention matrices — random initialisation (noise is expected)", y=1.01)
plt.tight_layout()
plt.show()

print("What to look for in a *trained* model:")
print("  • Diagonal band    — each token mostly attends to itself or neighbours.")
print("  • Vertical stripes — many tokens attend to one important anchor token.")
print("  • Isolated bright spots — a specific cross-token relationship (e.g. pronoun → referent).")
print()
print("All of the above emerge from training. Right now we only see noise.")

## Summary

| Step | Formula | What it adds |
|---|---|---|
| 4.2 | `S = X @ X.T` | Raw similarity between every token pair |
| 4.3 | `A = softmax(S)` | Converts scores to weights summing to 1 |
| 4.4 | `Y = A @ X` | Weighted average — tokens mix each other's info |
| 4.5 | `Q=X@W_Q`, `K=X@W_K`, `V=X@W_V` | Decouples three roles; adds learnable parameters |
| 4.6 | `scores / √d_k` | Prevents softmax saturation as dimension grows |

The function `scaled_dot_product_attention(Q, K, V)` we wrote in step 4.6 is the core primitive of every transformer ever built. The next section (Multi-Head Attention) simply runs it several times in parallel with different projections.